# 6.2 Custom Exceptions & Chaining

**Prerequisites:** 6.1 Exception Handling, 05 OOPs  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Designing an exception hierarchy for a library or package
- Naming conventions, and carrying structured data on an exception
- **`raise X from Y`** — explicit chaining
- `__cause__` vs `__context__`, and what 'During handling...' means
- **`raise ... from None`** to suppress a noisy inner cause
- **`Exception.add_note()`** (3.11) for context without wrapping
- **`ExceptionGroup` and `except*`** (3.11) for concurrent failures

---

## 1. Why define your own exceptions?

**6.1** showed the basic mechanics. The question this notebook answers is *when it is worth
it*, and *how to design the hierarchy*.

Consider a payments client that talks to an HTTP API. Internally it may raise
`requests.ConnectionError`, `json.JSONDecodeError`, `KeyError`, `ValueError`... Every one of
those leaks an implementation detail into your caller's code:

```python
try:
    charge_card(order)
except requests.ConnectionError:      # caller now depends on `requests`
    ...
except json.JSONDecodeError:          # ...and on your JSON parser
    ...
```

Swap `requests` for `httpx` and every caller breaks.

A custom hierarchy fixes this. The caller catches **your** exceptions, and you are free to
change what happens underneath.

### The three things a good hierarchy gives you

| Benefit | How |
|---|---|
| **A stable public contract** | Callers catch `PaymentError`, not `requests.ConnectionError` |
| **Choice of granularity** | Catch `PaymentError` broadly, or `CardDeclinedError` narrowly |
| **Structured data** | The exception carries the order id, the status code, the retry-after |

### Naming and structure conventions

- One **base exception per package**, inheriting from `Exception`. Everything else inherits
  from that base.
- Names end in **`Error`** (PEP 8). `PaymentError`, not `PaymentException` or `Check_price`.
- Inherit from a **specific builtin** only when your error genuinely *is* one — e.g. a
  validation error that callers would reasonably catch as `ValueError`.
- Give the base class a docstring; it is the documentation for the whole family.

In [ ]:
# ---- A hierarchy for a payments client ----

class PaymentError(Exception):
    """Base class for every error raised by this package.

    Callers can catch this to handle any payment failure.
    """


class PaymentConfigError(PaymentError):
    """The client was set up incorrectly - a programming error, not a runtime one."""


class PaymentTransportError(PaymentError):
    """The payment gateway could not be reached."""


class PaymentDeclinedError(PaymentError):
    """The gateway responded, and refused the charge."""

    def __init__(self, order_id: str, code: str, message: str = "") -> None:
        self.order_id = order_id            # structured data, not just a string
        self.code = code
        super().__init__(message or f"order {order_id} declined ({code})")

    def is_retryable(self) -> bool:
        """Behaviour on the exception itself - callers do not re-implement this."""
        return self.code in {"insufficient_funds", "issuer_unavailable"}


class CardExpiredError(PaymentDeclinedError):
    """A specific decline reason worth its own type."""


# ---- The caller chooses their granularity ----
def attempt(exc: PaymentError) -> str:
    try:
        raise exc
    except CardExpiredError as e:
        return f"ask for a new card (order {e.order_id})"
    except PaymentDeclinedError as e:
        return f"declined: {e.code}, retryable={e.is_retryable()}"
    except PaymentError as e:
        return f"generic payment failure: {type(e).__name__}"


for exc in [
    CardExpiredError("A-1001", "card_expired"),
    PaymentDeclinedError("A-1002", "insufficient_funds"),
    PaymentDeclinedError("A-1003", "fraud_suspected"),
    PaymentTransportError("gateway timed out"),
    PaymentConfigError("missing API key"),
]:
    print(f"  {type(exc).__name__:<24} -> {attempt(exc)}")


# ---- One catch covers the whole family ----
print("\nEverything is a PaymentError:")
for cls in (PaymentConfigError, PaymentTransportError, CardExpiredError):
    print(f"  {cls.__name__:<24} {issubclass(cls, PaymentError)}")

# ---- Structured data survives the raise ----
try:
    raise PaymentDeclinedError("A-1004", "insufficient_funds")
except PaymentDeclinedError as exc:
    print(f"\norder={exc.order_id} code={exc.code} retryable={exc.is_retryable()}")
    print("args:", exc.args)

---

## 2. Exception chaining: `raise X from Y`

When you catch a low-level error and raise your own, the original must not be lost —
otherwise you have thrown away the only evidence of what actually happened.

Python tracks two links on every exception:

| Attribute | Set by | Traceback says |
|---|---|---|
| `__context__` | **Automatically**, whenever you raise inside an `except` block | *"During handling of the above exception, another exception occurred"* |
| `__cause__` | **Explicitly**, with `raise X from Y` | *"The above exception was the direct cause of the following exception"* |

Both preserve the original traceback. The difference is **intent**:

- `__context__` means *"this happened while I was dealing with that"* — possibly incidental.
- `__cause__` means *"that is **why** this happened"* — a deliberate statement.

**Always use `from`** when you are deliberately translating one exception into another. It
costs four characters and makes the traceback say what you meant.

In [ ]:
import json, traceback


class ConfigError(Exception):
    """Raised when application configuration cannot be loaded."""


RAW_CONFIG = '{"timeout": "not-a-number"}'


# ---- (a) Implicit chaining: __context__ set automatically ----
def load_implicit(raw: str) -> dict:
    try:
        data = json.loads(raw)
        return {"timeout": int(data["timeout"])}
    except ValueError:
        raise ConfigError("could not load configuration")     # no `from`


# ---- (b) Explicit chaining: __cause__ set deliberately ----
def load_explicit(raw: str) -> dict:
    try:
        data = json.loads(raw)
        return {"timeout": int(data["timeout"])}
    except ValueError as exc:
        raise ConfigError("could not load configuration") from exc


# ---- (c) Suppressed: the inner cause is deliberately hidden ----
def load_suppressed(raw: str) -> dict:
    try:
        data = json.loads(raw)
        return {"timeout": int(data["timeout"])}
    except ValueError:
        raise ConfigError("timeout must be a whole number of seconds") from None


for func, label in [(load_implicit, "(a) implicit"),
                    (load_explicit, "(b) explicit 'from exc'"),
                    (load_suppressed, "(c) 'from None'")]:
    print("=" * 62)
    print(label)
    print("=" * 62)
    try:
        func(RAW_CONFIG)
    except ConfigError:
        print(traceback.format_exc())


# ---- Inspecting the links ----
try:
    load_explicit(RAW_CONFIG)
except ConfigError as exc:
    print("__cause__  :", repr(exc.__cause__))
    print("__context__:", repr(exc.__context__))
    print("suppressed :", exc.__suppress_context__)

try:
    load_suppressed(RAW_CONFIG)
except ConfigError as exc:
    print("\nwith 'from None':")
    print("__cause__  :", repr(exc.__cause__))
    print("__context__:", repr(exc.__context__), " <- still recorded")
    print("suppressed :", exc.__suppress_context__, " <- but hidden from the traceback")

### When to use `from None`

`raise ... from None` hides the inner exception from the traceback. Use it **only** when the
inner error is genuinely noise that would mislead the reader.

| Situation | Use |
|---|---|
| Translating a library error into yours | `from exc` — the cause is real information |
| Wrapping an error whose internals leak implementation detail the caller can't act on | `from None`, and put the useful part in your message |
| Re-raising the same error | bare `raise` (see **6.1**) |

⚠️ `from None` only suppresses the *display*. `__context__` is still set, so a logger or a
debugger can still reach the original. But a human reading the traceback will not see it —
so make sure your message contains everything they need.

---

## 3. `add_note()` — context without wrapping

> **Version note:** `Exception.add_note()` was added in **Python 3.11** (PEP 678).

Sometimes you want to attach context to an exception but **not** change its type — the
caller should still catch the same thing, you just want the traceback to say *which* record
failed, *which* file, *which* retry attempt.

Before 3.11 the options were bad: wrap it in a new exception (changing the type callers
must catch), or log separately (losing the connection to the traceback).

`add_note()` attaches free-text notes that print at the bottom of the traceback.

In [ ]:
import traceback


def parse_row(row: str, line_no: int, filename: str) -> int:
    try:
        return int(row)
    except ValueError as exc:
        # Add context WITHOUT changing the exception type
        exc.add_note(f"while parsing line {line_no} of {filename}")
        exc.add_note(f"offending content: {row!r}")
        raise


ROWS = ["10", "20", "thirty", "40"]

try:
    for i, row in enumerate(ROWS, start=1):
        parse_row(row, i, "readings.csv")
except ValueError:
    print(traceback.format_exc())


# The notes are just a list on the exception
try:
    parse_row("oops", 7, "data.csv")
except ValueError as exc:
    print("still a ValueError:", type(exc).__name__)
    print("__notes__:", exc.__notes__)


# ---- Accumulating notes as an exception propagates up the stack ----
def read_file(path: str) -> None:
    try:
        raise OSError("disk read failed")
    except OSError as exc:
        exc.add_note(f"path: {path}")
        raise


def load_dataset(name: str) -> None:
    try:
        read_file(f"/data/{name}.parquet")
    except OSError as exc:
        exc.add_note(f"dataset: {name}")
        exc.add_note("hint: check the volume is mounted")
        raise


try:
    load_dataset("sales_2024")
except OSError as exc:
    print("\nnotes accumulated on the way up:")
    for note in exc.__notes__:
        print("  -", note)

---

## 4. `ExceptionGroup` and `except*`

> **Version note:** `ExceptionGroup` and the `except*` syntax arrived in **Python 3.11**
> (PEP 654).

A normal `except` handles **one** exception. But some operations fail in **several ways at
once**:

- Running 50 downloads concurrently — 3 time out, 1 gets a 404, the rest succeed
- Validating a form — 4 fields are invalid, and the user wants to hear about all four
- Shutting down a service — two of the five subsystems fail to close cleanly

Raising only the first failure loses the others. `ExceptionGroup` bundles them, and
`except*` lets you handle each *kind* independently — a single group can match several
`except*` clauses.

### Syntax breakdown

```
try:
    raise ExceptionGroup("msg", [ValueError(...), TypeError(...)])
except* ValueError as eg:      <- eg is a GROUP containing only the ValueErrors
    ...
except* TypeError as eg:       <- runs TOO, for the TypeErrors
    ...
```

Note `except*` binds a **group**, not a single exception — so you iterate `eg.exceptions`.

In [ ]:
import traceback


# ---- Validating a form: report every problem, not just the first ----
def validate(form: dict) -> None:
    errors = []

    if not form.get("email", "").count("@"):
        errors.append(ValueError("email must contain @"))
    if len(form.get("password", "")) < 8:
        errors.append(ValueError("password must be at least 8 characters"))
    age = form.get("age")
    if not isinstance(age, int):
        errors.append(TypeError(f"age must be an integer, got {type(age).__name__}"))
    elif age < 0:
        errors.append(ValueError("age must not be negative"))

    if errors:
        raise ExceptionGroup("form validation failed", errors)


form = {"email": "not-an-email", "password": "short", "age": "twenty"}

try:
    validate(form)
except* ValueError as group:
    print("Value problems:")
    for exc in group.exceptions:
        print("  -", exc)
except* TypeError as group:
    print("Type problems:")
    for exc in group.exceptions:
        print("  -", exc)

print("\nBoth handlers ran - one group, matched by two clauses.\n")


# ---- What the traceback looks like ----
try:
    validate(form)
except ExceptionGroup:
    print(traceback.format_exc())


# ---- Groups nest, and you can still catch the whole thing ----
def shutdown() -> None:
    raise ExceptionGroup("shutdown failed", [
        OSError("could not close socket"),
        ExceptionGroup("cache errors", [
            RuntimeError("flush timed out"),
            RuntimeError("eviction thread hung"),
        ]),
    ])


try:
    shutdown()
except* OSError as group:
    print("OS errors  :", [str(e) for e in group.exceptions])
except* RuntimeError as group:
    print("Runtime    :", [str(e) for e in group.exceptions], " <- found inside the nested group")


# ---- A plain `except` still catches the group as a whole ----
try:
    shutdown()
except ExceptionGroup as group:
    print("\ncaught whole group:", group.message, "|", len(group.exceptions), "top-level members")


# ---- Where you will actually meet this: asyncio.TaskGroup (see folder 12) ----
print("""
asyncio.TaskGroup (3.11+) raises an ExceptionGroup when several concurrent
tasks fail. That is the most common way you will encounter this in real code.
""")

---

## Common Mistakes & Pitfalls

1. **Raising a new exception without `from`.** The traceback then says 'During handling of the above exception...' — which reads like an accident rather than a deliberate translation.
2. **Using `from None` to hide an error you did not understand.** It suppresses the evidence you would have needed.
3. **Inheriting from `BaseException`.** Custom exceptions inherit from `Exception`. `BaseException` is for interpreter-level control flow only.
4. **No common base class.** Without one, callers must enumerate every exception your package can raise — and you can never add a new one without breaking them.
5. **Overriding `__str__` to reimplement what `Exception` already does.** Passing the message to `super().__init__()` is enough.
6. **Overriding `__init__` without calling `super().__init__()`.** `exc.args` ends up empty and pickling breaks.
7. **A hierarchy with twenty exception types.** Callers cannot use that granularity. Three to six is usually right.
8. **Assuming `except*` catches a plain exception.** It does not match non-grouped exceptions the way you might expect — and you cannot mix `except` and `except*` in the same `try`.

## Best Practices

- Define **one base exception per package**, inheriting from `Exception`, and derive everything from it.
- Name them `...Error` (PEP 8).
- **Always `raise ... from exc`** when translating one exception into another.
- Put **structured data** on the exception (ids, codes, retry-after) rather than formatting it all into the message string.
- Give exceptions **methods** where it helps — `is_retryable()` beats making every caller reimplement that check.
- Use `add_note()` to add context as an exception travels up, instead of re-wrapping it.
- Use `ExceptionGroup` when several independent operations can fail together.
- Document which exceptions your public functions raise — it is part of the signature.

## Practice Exercises

Try these before moving on.

1. Design a three-level exception hierarchy for a file-sync tool and write a caller that handles two levels differently.
2. Wrap a `json.JSONDecodeError` in your own `ConfigError`, once with `from` and once without. Compare the tracebacks.
3. Write a function that uses `from None` appropriately, and justify the choice.
4. Add `add_note()` calls at three levels of a call stack and print the accumulated notes.
5. Write a validator that collects all field errors into an `ExceptionGroup`, then handle `ValueError` and `TypeError` separately with `except*`.
6. Give a custom exception a `retry_after` attribute and write a retry loop that uses it.
7. Show that `except*` and `except` cannot be mixed in one `try` statement.
8. Take a function from **6.1** that raises a builtin and convert it to raise a domain-specific exception instead. Did the caller get simpler?